# GameTheory-20 : Capstone — la sous-série Coalitions en une heure

**Série** : GameTheory — jumeau C# du carnet Python
**Source du portage** : [GameTheory-20-Capstone-Coalitions-Python.ipynb](GameTheory-20-Capstone-Coalitions-Python.ipynb)
**Kernel** : `.net-csharp` (BCL seule, aucun NuGet) — mêmes témoins recalculés : Banzhaf, cycles et chaînes altruistes, Kuhn-Munkres et son certificat dual, inventaire du lake Lean

**Navigation** : [<< 17-MultiAgent-RL](GameTheory-17-MultiAgent-RL-Python.ipynb) | [Index](README.md) | [Capstone 21 >>](GameTheory-21-Capstone-Choix-Social-Python.ipynb) · sous-série [Coalitions](Coalitions/README.md) | [Version Python](GameTheory-20-Capstone-Coalitions-Python.ipynb)

**Side tracks** : ce carnet EST l'escalier vers la sous-série [Coalitions](Coalitions/README.md) — il la présente et n'en remplace aucun carnet.

Ce carnet est le **capstone** de la sous-série Coalitions, consacrée à une question que la théorie des jeux coopératifs pose depuis von Neumann et Morgenstern : **combien vaut une coalition, et à quel niveau de garantie**. La sous-série y répond par paliers — un indice calculé, une borne no-regret, une institution réelle, un optimum dual, une preuve formelle — et ce capstone les fait dialoguer sur les mêmes objets : chaque expérience confronte **deux niveaux de garantie** sur **un seul exemple**.

Il suppose lu le parcours principal de la série (théorie des jeux non-coopérative, équilibres) et referme la branche coopération ouverte par les carnets [01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) à [05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb).

## Le fil en une phrase

Poids n'est pas pouvoir ; une coalition ne vaut pas son poids mais ce que ses membres pivots peuvent obtenir ; et la valeur d'une coalition se décline en **garanties de force croissante** : mesurée (simulation), prouvée (certificat vérifiable), ou démontrée (théorème formel).


## 1. Ce que la sous-série démontre

| Carnet | Ce qu'il démontre | Niveau de garantie |
|---|---|---|
| [Coalitions-01 — Pouvoir coalitionnel par SMT](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) | poids ≠ pouvoir : trois indices contrefactuels (Shapley-Shubik, Banzhaf, Deegan-Packel), puis énumération SMT en seconde voie | indice **calculé** (énumération exacte) |
| [Coalitions-02 — Assistance Games](Coalitions/Coalitions-02-Assistance-Games-Python.ipynb) | coopération humain-agent : oracle en hindsight contre apprenant no-regret EXP3, bench √T contre T^(3/4) | **borne** no-regret (garantie asymptotique) |
| [Coalitions-03 — Échange de reins](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb) | le matching comme institution : cycles du graphe de compatibilité, chaînes de donneurs altruistes, arbitrage cardinalité/équité | institution **réelle** (données et contraintes du monde) |
| [Coalitions-04 — Munkres et affectation](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb) | l'affectation optimale : Kuhn-Munkres from scratch, lecture par dualité, jeu d'affectation de Shapley-Shubik, deux notions d'optimum | optimum **dual** (certificat vérifiable) |
| [Coalitions-05 — Lean Affectation](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) | le lake `assignment_lean` par son certificat : quatre modules, valeur d'une affectation, ce que la preuve garantit | preuve **formelle** (théorème machine-vérifié) |

Le fil que la sous-série tisse : une même question — combien vaut coopérer — reçoit des réponses de force croissante. Le capstone fait tenir trois de ces niveaux dans la même fenêtre d'expérience.

## 2. Ce que ce notebook suppose

- **Un jeu coopératif pondéré** : des joueurs dotés de poids, un quota, des coalitions gagnantes — posé par [Coalitions-01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb), dont nous reprenons le noyau tel quel.
- **Un graphe de compatibilité dirigé** entre paires donneur-receveur — posé par [Coalitions-03](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb).
- **Une matrice de coûts et sa dualité** : les potentiels duaux `u_i + v_j <= c_ij` — posés par [Coalitions-04](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb), dont nous reprenons l'algorithme de Kuhn-Munkres intégralement.
- Les sources du lake `assignment_lean` sont **lues** (noms des théorèmes, structure des modules), jamais ré-exécutées ici : la preuve de build vit dans [Coalitions-05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) et sa CI.

Aucun prérequis hors la série : la **BCL .NET suffit** — aucun paquet NuGet, aucune dépendance externe, aucun service en ligne. Le carnet s'exécute de bout en bout, exercices non complétés compris.


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.IO;
using System.Linq;
using System.Text.RegularExpressions;

// En C#, un type se declare apres les instructions de premiere ligne : on
// regroupe donc ici les instructions de la cellule, puis les types qu'elles
// utilisent (ils sont resolus independent de l'ordre dans le fichier).

var racine = Capstone.FindRepoRoot();
Console.WriteLine("imports OK — BCL seule (aucun NuGet, aucune dependance externe)");
Console.WriteLine("lac assignment_lean : trouve (MyIA.AI.Notebooks/GameTheory/assignment_lean)");

// Outils de mise en table partages avec le jumeau Python : on reproduit la
// mise en forme du tableau pandas (index a gauche, valeurs alignees a droite)
// pour que les deux carnets affichent les memes tableaux.
public static class Capstone
{
    /// Remontee d'ancetres jusqu'au lac assignment_lean (pattern Coalitions-03,
    /// independant du repertoire courant de la session).
    public static DirectoryInfo FindRepoRoot()
    {
        var p = new DirectoryInfo(Directory.GetCurrentDirectory());
        while (p != null)
        {
            if (Directory.Exists(Path.Combine(p.FullName, "MyIA.AI.Notebooks", "GameTheory", "assignment_lean"))) return p;
            p = p.Parent;
        }
        throw new DirectoryNotFoundException("racine du depot introuvable depuis " + Directory.GetCurrentDirectory());
    }
}

public static class Table
{
    /// Rendu d'un tableau, colonne par colonne.
    /// Une colonne homogene (toutes valeurs numeriques, ou toutes chaines) aligne
    /// son entete a droite ; une colonne mixte aligne l'entete a gauche et les
    /// valeurs a droite. La largeur d'une colonne est celle de son plus long
    /// element (entete comprise).
    public static List<string> RendreTable(string[] entetes, string[][] lignes, int separateur, string[] indexLabels, string indexEntete)
    {
        int nCol = entetes.Length;
        var largeur = new int[nCol];
        var enteteGauche = new bool[nCol];
        for (int j = 0; j < nCol; j++)
        {
            int maxVal = 0;
            bool tousNumeriques = true, tousChaines = true;
            foreach (var ligne in lignes)
            {
                string v = ligne[j];
                if (v.Length > maxVal) maxVal = v.Length;
                if (double.TryParse(v, NumberStyles.Float, CultureInfo.InvariantCulture, out _)) tousChaines = false;
                else tousNumeriques = false;
            }
            largeur[j] = Math.Max(entetes[j].Length, maxVal);
            enteteGauche[j] = !(tousNumeriques || tousChaines);
        }

        string sep = new string(' ', separateur);
        var sortie = new List<string>();

        var enteteLigne = new string[nCol];
        for (int j = 0; j < nCol; j++)
            enteteLigne[j] = enteteGauche[j] ? entetes[j].PadRight(largeur[j]) : entetes[j].PadLeft(largeur[j]);

        if (indexLabels == null)
        {
            sortie.Add(string.Join(sep, enteteLigne));
            foreach (var l in lignes)
            {
                var cells = new string[nCol];
                for (int j = 0; j < nCol; j++) cells[j] = l[j].PadLeft(largeur[j]);
                sortie.Add(string.Join(sep, cells));
            }
            return sortie;
        }

        int wIndex = Math.Max(indexEntete.Length, indexLabels.Max(l => l.Length));
        sortie.Add(indexEntete.PadRight(wIndex) + sep + string.Join(sep, enteteLigne));
        for (int i = 0; i < lignes.Length; i++)
        {
            var cells = new string[nCol];
            for (int j = 0; j < nCol; j++) cells[j] = lignes[i][j].PadLeft(largeur[j]);
            sortie.Add(indexLabels[i].PadRight(wIndex) + sep + string.Join(sep, cells));
        }
        return sortie;
    }
}


The below script needs to be able to find the current output cell; this is an easy method to get it.

imports OK — BCL seule (aucun NuGet, aucune dependance externe)


lac assignment_lean : trouve (MyIA.AI.Notebooks/GameTheory/assignment_lean)


## 3. Expérience 1 — Le président pénalisé

*Ancrée dans [Coalitions-01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb), dont nous reprenons le noyau en place.*

Un président détient le plus gros poids d'une assemblée. La question contrefactuelle de Coalitions-01 : si son parti obtient **un vote de quota de plus** à faire adopter, que devient son pouvoir réel ? L'indice de Banzhaf compte pour chaque joueur les coalitions où son adhésion fait basculer la victoire — les *swings* — et normalise. C'est la garantie de niveau 1 : une énumération exacte, vérifiable par le SMT de 01 en seconde voie.

In [2]:
// Noyau repris de Coalitions-01 (VotingGame), compacte pour le capstone.
var president = new VotingGame(new int[] { 3, 2, 1 }, 4, new string[] { "President", "B", "C" });
Console.WriteLine("Jeu de depart [4; 3, 2, 1] — coalitions gagnantes minimales :");

static string coalitions(List<HashSet<int>> coalitions, string[] names)
{
    return "[" + string.Join(", ", coalitions.Select(s => "[" + string.Join(", ", s.OrderBy(i => i).Select(i => "'" + names[i] + "'")) + "]")) + "]";
}

Console.WriteLine(coalitions(president.MinimalWinning(), president.Names));

// Assertion : les coalitions gagnantes minimales mesurees sont celles de la source.
var attendues = new List<string> { "['President', 'B']", "['President', 'C']" };
var mesurees = president.MinimalWinning().Select(s => "[" + string.Join(", ", s.OrderBy(i => i).Select(i => "'" + president.Names[i] + "'")) + "]").ToList();
if (!mesurees.SequenceEqual(attendues))
    throw new Exception("coalitions minimales differentes de la source : " + string.Join(" | ", mesurees));

public sealed class VotingGame
{
    public int[] Weights { get; }
    public int Quota { get; }
    public string[] Names { get; }

    public VotingGame(int[] weights, int quota, string[] names)
    {
        Weights = weights;
        Quota = quota;
        Names = names;
    }

    public int Weight(IEnumerable<int> coalition)
    {
        int s = 0;
        foreach (var i in coalition) s += Weights[i];
        return s;
    }

    public bool Wins(HashSet<int> coalition) => Weight(coalition) >= Quota;

    public static IEnumerable<int[]> Combinations(int n, int size)
    {
        if (size == 0) { yield return new int[0]; yield break; }
        var idx = Enumerable.Range(0, size).ToArray();
        while (true)
        {
            yield return (int[])idx.Clone();
            int i = size - 1;
            while (i >= 0 && idx[i] == n - size + i) i--;
            if (i < 0) yield break;
            idx[i]++;
            for (int j = i + 1; j < size; j++) idx[j] = idx[j - 1] + 1;
        }
    }

    public List<HashSet<int>> Coalitions()
    {
        var res = new List<HashSet<int>>();
        for (int size = 0; size <= Weights.Length; size++)
            foreach (var c in Combinations(Weights.Length, size)) res.Add(new HashSet<int>(c));
        return res;
    }

    public List<HashSet<int>> MinimalWinning()
    {
        var res = new List<HashSet<int>>();
        foreach (var s in Coalitions())
        {
            if (!Wins(s)) continue;
            bool minimal = true;
            foreach (var i in s.ToArray())
            {
                var copy = new HashSet<int>(s);
                copy.Remove(i);
                if (Wins(copy)) { minimal = false; break; }
            }
            if (minimal) res.Add(s);
        }
        return res;
    }

    public bool IsDummy(int i)
    {
        foreach (var s in Coalitions())
        {
            if (s.Contains(i)) continue;
            var copy = new HashSet<int>(s);
            copy.Add(i);
            if (Wins(copy) != Wins(s)) return false;
        }
        return true;
    }
}

public static class Noyau01
{
    public static Dictionary<string, int> SwingCounts(VotingGame game)
    {
        var counts = new Dictionary<string, int>();
        for (int i = 0; i < game.Weights.Length; i++) counts[game.Names[i]] = 0;
        foreach (var s in game.Coalitions())
        {
            if (game.Wins(s)) continue;
            for (int i = 0; i < game.Weights.Length; i++)
            {
                if (s.Contains(i)) continue;
                var copy = new HashSet<int>(s);
                copy.Add(i);
                if (game.Wins(copy)) counts[game.Names[i]]++;
            }
        }
        return counts;
    }

    /// Banzhaf normalise : swings du joueur sur total des swings (cf. Coalitions-01, section 2).
    public static Dictionary<string, double> Banzhaf(VotingGame game)
    {
        var swings = SwingCounts(game);
        int total = swings.Values.Sum();
        var res = new Dictionary<string, double>();
        foreach (var nom in game.Names) res[nom] = (double)swings[nom] / total;
        return res;
    }
}


Jeu de depart [4; 3, 2, 1] — coalitions gagnantes minimales :


[['President', 'B'], ['President', 'C']]


In [3]:
// L'experience : un seul vote de quota de plus (4 -> 5), memes poids.
var avant = new VotingGame(new int[] { 3, 2, 1 }, 4, new string[] { "President", "B", "C" });
var apres = new VotingGame(new int[] { 3, 2, 1 }, 5, new string[] { "President", "B", "C" });

var jeux = new[] { avant, apres };
var entetes = new string[jeux.Length];
for (int j = 0; j < jeux.Length; j++)
    entetes[j] = "[" + jeux[j].Quota + "; " + string.Join(", ", jeux[j].Weights) + "] (quota " + jeux[j].Quota + ")";

var banzhafs = jeux.Select(Noyau01.Banzhaf).ToArray();
var labels = new List<string>();
var lignes = new List<string[]>();
foreach (var nom in avant.Names)
{
    labels.Add("poids " + nom);
    lignes.Add(jeux.Select(j => j.Weights[Array.IndexOf(j.Names, nom)].ToString(CultureInfo.InvariantCulture)).ToArray());
}
foreach (var nom in avant.Names)
{
    labels.Add("banzhaf " + nom);
    lignes.Add(banzhafs.Select(b => b[nom].ToString("0.0###", CultureInfo.InvariantCulture)).ToArray());
}
labels.Add("C nul");
lignes.Add(jeux.Select(j => j.IsDummy(2) ? "True" : "False").ToArray());

foreach (var l in Table.RendreTable(entetes, lignes.ToArray(), 1, labels.ToArray(), "jeu"))
    Console.WriteLine(l);

// Assertions : les valeurs mesurees sont celles de la source.
if (lignes[0][0] != "3" || lignes[0][1] != "3") throw new Exception("poids President : " + string.Join(",", lignes[0]));
if (lignes[3][0] != "0.6" || lignes[3][1] != "0.5") throw new Exception("banzhaf President : " + string.Join(",", lignes[3]));
if (lignes[6][0] != "False" || lignes[6][1] != "True") throw new Exception("C nul : " + string.Join(",", lignes[6]));


jeu               [4; 3, 2, 1] (quota 4) [5; 3, 2, 1] (quota 5)


poids President                        3                      3


poids B                                2                      2


poids C                                1                      1


banzhaf President                    0.6                    0.5


banzhaf B                            0.2                    0.5


banzhaf C                            0.2                    0.0


C nul                              False                   True


### Lecture du résultat — un vote de quota, trois destins

La mesure ci-dessus est celle de Coalitions-01, resserrée sur un seul levier. Au quota 4, le président (poids 3, soit 50 % de l'assemblée) domine : son Banzhaf vaut 0,6 contre 0,2 pour chacun des autres. Au quota 5, **un seul vote de plus à réunir** :

- le président tombe de 0,6 à **0,5** : sa moitié de poids ne lui achète plus que la moitié du pouvoir ;
- B monte de 0,2 à **0,5** : à poids inférieur, pouvoir égal — les coalitions gagnantes passent toutes par l'un *ou* l'autre ;
- C tombe à **0,0** et devient nul (`is_dummy` vrai) : son poids existe, son pouvoir a disparu.

C'est le cœur de « poids n'est pas pouvoir » : la valeur d'un membre d'une coalition ne se lit pas dans son poids mais dans ses **swings** — les situations où son adhésion bascule la coalition. Coalitions-01 pousse cette énumération jusqu'à la garantie SMT (`UNSAT`/`SAT` sur les jeux croisés) ; le capstone s'arrête à l'indice exact, et renvoie au carnet source pour la seconde voie.

## 4. Expérience 2 — Cycle contre chaîne

*Ancrée dans [Coalitions-03](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb), dont nous reprenons la construction du graphe.*

L'échange de reins est l'institution réelle de la sous-série : des paires donneur-receveur incompatibles entre elles cherchent un échange. Deux mécanismes s'affrontent : les **cycles** entre paires (chacune donne à la suivante, la boucle se referme) et les **chaînes** amorcées par un donneur altruiste (la chaîne peut s'arrêter sans refermer de boucle — un donneur « terminal » peut donner à la liste d'attente). La question de 03 : que change un altruiste, en cardinalité **et** en équité ? C'est la garantie de niveau 2 : une institution avec les contraintes du monde, où la couverture se mesure, ne se démontre pas.

Le paramètre qui décide est le **groupe du donneur altruiste** : il détermine quelles paires il peut atteindre, et donc ce que la chaîne ajoute réellement aux cycles — la cellule de mesure le fait varier sur la même instance.

In [4]:
// Noyau repris de Coalitions-03 : graphe de compatibilite, cycles, chaines altruistes.
// Instance fixee (meme semence que 03, reproductible) : le generateur ci-dessous
// reproduit le flux de random.Random(12240) de CPython terme a terme, donc les
// six paires mesurees sont celles du carnet Python.
var rng = new Mt19937(12240);
var groupes = new[] { "O", "A", "B", "AB" };
var paires = new (string donneur, string receveur)[6];
for (int i = 0; i < paires.Length; i++) paires[i] = (rng.Choice(groupes), rng.Choice(groupes));
var altruistes = new[] { "B", "B" };

var gSans = Noyau03.ConstruireGrapheAvecAltruistes(paires, Array.Empty<string>());
var gAvec = Noyau03.ConstruireGrapheAvecAltruistes(paires, altruistes);

Console.WriteLine("Paires (donneur, receveur) : [" + string.Join(", ", paires.Select(p => "('" + p.donneur + "', '" + p.receveur + "')")) + "]");
Console.WriteLine("Altruistes : [" + string.Join(", ", altruistes.Select(a => "'" + a + "'")) + "]");
Console.WriteLine("Aretes sans altruiste : " + Noyau03.NombreAretes(gSans) + " | avec : " + Noyau03.NombreAretes(gAvec));

// Assertion : les six paires mesurees sont celles de la source.
var pairesAttendues = new[] { "('B', 'B')", "('AB', 'O')", "('B', 'B')", "('O', 'AB')", "('A', 'AB')", "('AB', 'B')" };
var pairesMesurees = paires.Select(p => "('" + p.donneur + "', '" + p.receveur + "')").ToArray();
if (!pairesMesurees.SequenceEqual(pairesAttendues))
    throw new Exception("paires differentes de la source : " + string.Join(", ", pairesMesurees));
if (Noyau03.NombreAretes(gSans) != 9 || Noyau03.NombreAretes(gAvec) != 15)
    throw new Exception("nombre d'aretes different de la source");

/// Generateur MT19937 : meme initialisation et meme consommation que
/// random.Random(seed) de CPython (init_by_array + getrandbits avec rejet).
public sealed class Mt19937
{
    const int N = 624;
    const int M = 397;
    const uint MatrixA = 0x9908b0dfU;
    const uint UpperMask = 0x80000000U;
    const uint LowerMask = 0x7fffffffU;

    readonly uint[] _mt = new uint[N];
    int _mti;

    public Mt19937(int seed) { InitByArray(new uint[] { unchecked((uint)Math.Abs((long)seed)) }); }

    void InitGenrand(uint s)
    {
        _mt[0] = s;
        for (int i = 1; i < N; i++)
            _mt[i] = unchecked(1812433253U * (_mt[i - 1] ^ (_mt[i - 1] >> 30)) + (uint)i);
        _mti = N;
    }

    void InitByArray(uint[] key)
    {
        InitGenrand(19650218U);
        int i = 1, j = 0;
        int k = Math.Max(N, key.Length);
        for (; k > 0; k--)
        {
            _mt[i] = unchecked((uint)((_mt[i] ^ ((_mt[i - 1] ^ (_mt[i - 1] >> 30)) * 1664525U)) + key[j] + (uint)j));
            i++;
            j++;
            if (i >= N) { _mt[0] = _mt[N - 1]; i = 1; }
            if (j >= key.Length) j = 0;
        }
        for (k = N - 1; k > 0; k--)
        {
            _mt[i] = unchecked((uint)((_mt[i] ^ ((_mt[i - 1] ^ (_mt[i - 1] >> 30)) * 1566083941U)) - (uint)i));
            i++;
            if (i >= N) { _mt[0] = _mt[N - 1]; i = 1; }
        }
        _mt[0] = 0x80000000U;
    }

    public uint NextUInt32()
    {
        uint y;
        if (_mti >= N)
        {
            int kk;
            for (kk = 0; kk < N - M; kk++)
            {
                y = (_mt[kk] & UpperMask) | (_mt[kk + 1] & LowerMask);
                _mt[kk] = _mt[kk + M] ^ (y >> 1) ^ ((y & 1U) != 0U ? MatrixA : 0U);
            }
            for (; kk < N - 1; kk++)
            {
                y = (_mt[kk] & UpperMask) | (_mt[kk + 1] & LowerMask);
                _mt[kk] = _mt[kk + (M - N)] ^ (y >> 1) ^ ((y & 1U) != 0U ? MatrixA : 0U);
            }
            y = (_mt[N - 1] & UpperMask) | (_mt[0] & LowerMask);
            _mt[N - 1] = _mt[M - 1] ^ (y >> 1) ^ ((y & 1U) != 0U ? MatrixA : 0U);
            _mti = 0;
        }
        y = _mt[_mti++];
        y ^= (y >> 11);
        y ^= (y << 7) & 0x9d2c5680U;
        y ^= (y << 15) & 0xefc60000U;
        y ^= (y >> 18);
        return y;
    }

    public ulong GetRandBits(int k)
    {
        int words = (k - 1) / 32 + 1;
        ulong result = 0;
        for (int i = 0; i < words; i++)
        {
            uint r = NextUInt32();
            int bits = k - i * 32;
            if (bits < 32) r >>= (32 - bits);
            result |= (ulong)r << (i * 32);
        }
        return result;
    }

    /// _randbelow de CPython : tire k bits et rejette tant que le resultat depasse n.
    public int RandBelow(int n)
    {
        int k = 32 - BitsEnTete((uint)n);
        ulong r = GetRandBits(k);
        while (r >= (ulong)n) r = GetRandBits(k);
        return (int)r;
    }

    static int BitsEnTete(uint v)
    {
        if (v == 0) return 32;
        int c = 0;
        while ((v & 0x80000000U) == 0) { v <<= 1; c++; }
        return c;
    }

    public T Choice<T>(IReadOnlyList<T> seq) => seq[RandBelow(seq.Count)];
}

/// Graphe dirige de compatibilite entre paires donneur-receveur et donneurs altruistes.
public sealed class Graphe
{
    public sealed class Noeud
    {
        public string Role { get; }
        public string Donneur { get; }
        public string Receveur { get; }
        public Noeud(string role, string donneur, string receveur) { Role = role; Donneur = donneur; Receveur = receveur; }
    }

    readonly Dictionary<string, Noeud> _noeuds = new Dictionary<string, Noeud>();
    readonly Dictionary<string, List<string>> _arcs = new Dictionary<string, List<string>>();

    public void Add(string id, string role, string donneur, string receveur)
    {
        _noeuds[id] = new Noeud(role, donneur, receveur);
        if (!_arcs.ContainsKey(id)) _arcs[id] = new List<string>();
    }

    public void AddArc(string a, string b) => _arcs[a].Add(b);

    public IReadOnlyList<string> Successeurs(string id) => _arcs[id];

    public IEnumerable<string> Noeuds => _noeuds.Keys;

    public Noeud Info(string id) => _noeuds[id];

    public int NombreAretes() => _arcs.Values.Sum(v => v.Count);

    public Graphe Copy()
    {
        var g = new Graphe();
        foreach (var kv in _noeuds) g.Add(kv.Key, kv.Value.Role, kv.Value.Donneur, kv.Value.Receveur);
        foreach (var kv in _arcs) foreach (var v in kv.Value) g.AddArc(kv.Key, v);
        return g;
    }

    public void Remove(IEnumerable<string> ids)
    {
        foreach (var id in ids) { _noeuds.Remove(id); _arcs.Remove(id); }
        foreach (var kv in _arcs) kv.Value.RemoveAll(x => !_noeuds.ContainsKey(x));
    }
}

public static class Noyau03
{
    public static Graphe ConstruireGrapheAvecAltruistes((string donneur, string receveur)[] paires, string[] altruistes)
    {
        var g = new Graphe();
        for (int i = 0; i < paires.Length; i++) g.Add("P" + i, "paire", paires[i].donneur, paires[i].receveur);
        for (int j = 0; j < altruistes.Length; j++) g.Add("A" + j, "altruiste", altruistes[j], null);
        for (int j = 0; j < altruistes.Length; j++)
            for (int i = 0; i < paires.Length; i++)
                if (altruistes[j] == paires[i].receveur) g.AddArc("A" + j, "P" + i);
        for (int i = 0; i < paires.Length; i++)
            for (int k = 0; k < paires.Length; k++)
                if (i != k && paires[i].donneur == paires[k].receveur) g.AddArc("P" + i, "P" + k);
        return g;
    }

    public static int NombreAretes(Graphe g) => g.NombreAretes();

    /// Couverture gloutonne : cycles de paires disjoints, les plus longs d'abord.
    public static List<List<string>> CouvertureParCycles(Graphe g, int kMax)
    {
        var noeuds = g.Noeuds.Where(n => g.Info(n).Role == "paire").OrderBy(n => n, StringComparer.Ordinal).ToList();
        var candidats = new List<List<string>>();
        foreach (var depart in noeuds)
        {
            var chemin = new List<string> { depart };
            EnumCycles(g, noeuds, kMax, noeuds.IndexOf(depart), chemin, new HashSet<string> { depart }, candidats);
        }
        var tries = candidats.OrderByDescending(c => c.Count).ToList();
        var pris = new List<List<string>>();
        var prisDans = new HashSet<string>();
        foreach (var c in tries)
        {
            if (c.Any(x => prisDans.Contains(x))) continue;
            pris.Add(c);
            foreach (var x in c) prisDans.Add(x);
        }
        return pris;
    }

    static void EnumCycles(Graphe g, List<string> noeuds, int kMax, int indexDepart, List<string> chemin, HashSet<string> surChemin, List<List<string>> acc)
    {
        string courant = chemin[chemin.Count - 1];
        if (chemin.Count > 1 && g.Successeurs(courant).Contains(chemin[0]))
        {
            var cycle = Canonique(chemin);
            if (!acc.Any(c => c.SequenceEqual(cycle))) acc.Add(cycle);
            return;
        }
        if (chemin.Count >= kMax) return;
        foreach (var succ in g.Successeurs(courant))
        {
            if (surChemin.Contains(succ)) continue;
            int index = noeuds.IndexOf(succ);
            if (index < indexDepart) continue;
            chemin.Add(succ);
            surChemin.Add(succ);
            EnumCycles(g, noeuds, kMax, indexDepart, chemin, surChemin, acc);
            chemin.RemoveAt(chemin.Count - 1);
            surChemin.Remove(succ);
        }
    }

    /// Cycle canonique : plus petit noeud en tete, puis orientation minimale.
    static List<string> Canonique(List<string> cycle)
    {
        int meilleur = 0;
        for (int i = 1; i < cycle.Count; i++)
            if (string.CompareOrdinal(cycle[i], cycle[meilleur]) < 0) meilleur = i;
        var rot = new List<string>();
        for (int i = 0; i < cycle.Count; i++) rot.Add(cycle[(meilleur + i) % cycle.Count]);
        var inverse = new List<string> { rot[0] };
        for (int i = rot.Count - 1; i >= 1; i--) inverse.Add(rot[i]);
        for (int i = 1; i < rot.Count; i++)
        {
            int c = string.CompareOrdinal(rot[i], inverse[i]);
            if (c != 0) return c < 0 ? rot : inverse;
        }
        return rot;
    }

    /// Chaines demarrees par un altruiste (DFS bornee), reprises de 03 tel quel.
    /// Le graphe recu est celui sur lequel la chaine a le DROIT de courir : pour un
    /// plan combine cycles + chaine, c'est le graphe residuel (paires non servies).
    public static List<List<string>> EnumererChaines(Graphe g, int kMax)
    {
        var chaines = new List<List<string>>();
        foreach (var source in g.Noeuds.Where(n => g.Info(n).Role == "altruiste"))
            DfsChaines(g, source, new List<string> { source }, 0, kMax, chaines);
        return chaines;
    }

    static void DfsChaines(Graphe g, string noeud, List<string> chemin, int profondeur, int kMax, List<List<string>> acc)
    {
        if (profondeur >= kMax) { acc.Add(new List<string>(chemin)); return; }
        foreach (var succ in g.Successeurs(noeud))
        {
            if (chemin.Contains(succ)) continue;
            chemin.Add(succ);
            DfsChaines(g, succ, chemin, profondeur + 1, kMax, acc);
            chemin.RemoveAt(chemin.Count - 1);
        }
        if (profondeur >= 1) acc.Add(new List<string>(chemin));
    }

    /// Couverture par cycles d'abord, puis chaine sur le graphe RESIDUEL.
    /// Contrainte du monde : un donneur ne donne qu'une fois. Une paire servie par un
    /// cycle ne peut donc pas etre traversee par une chaine -- enumerer les chaines sur
    /// le graphe complet fabriquerait des plans infaisables ou une meme paire donne deux fois.
    public static (List<List<string>> cycles, List<string> meilleure, HashSet<string> servies) CouvertureCyclesPuisChaine(
        (string donneur, string receveur)[] paires, string[] altruistes, int kCycles, int kChaine)
    {
        var g = ConstruireGrapheAvecAltruistes(paires, altruistes);
        var cycles = CouvertureParCycles(g, kCycles);
        var servies = new HashSet<string>();
        foreach (var c in cycles) foreach (var x in c) servies.Add(x);

        var residuel = g.Copy();
        residuel.Remove(servies);
        var chaines = EnumererChaines(residuel, kChaine);

        var meilleure = new List<string>();
        int gainMax = -1, longueurMax = -1;
        foreach (var ch in chaines)
        {
            int gain = ch.Count(x => x.StartsWith("P") && !servies.Contains(x));
            if (gain > gainMax || (gain == gainMax && ch.Count > longueurMax))
            {
                meilleure = new List<string>(ch);
                gainMax = gain;
                longueurMax = ch.Count;
            }
        }
        foreach (var x in meilleure) if (x.StartsWith("P")) servies.Add(x);
        return (cycles, meilleure, servies);
    }

    /// Formatage des noeuds a la maniere du carnet source : ('P', 0).
    public static string Tuple(string id) => "('" + id.Substring(0, 1) + "', " + id.Substring(1) + ")";
}


Paires (donneur, receveur) : [('B', 'B'), ('AB', 'O'), ('B', 'B'), ('O', 'AB'), ('A', 'AB'), ('AB', 'B')]


Altruistes : ['B', 'B']


Aretes sans altruiste : 9 | avec : 15


In [5]:
// La mesure : les cycles d'abord, puis la chaine sur le graphe RESIDUEL.
// Le groupe du donneur altruiste est le seul parametre qui varie.
foreach (var groupe in groupes)
{
    var mesure = Noyau03.CouvertureCyclesPuisChaine(paires, new[] { groupe, groupe }, 3, 4);
    string cyclesTxt = string.Join(" | ", mesure.cycles.Select(c => string.Join(",", c.Select(x => "P" + x.Substring(1)))));
    string chaineTxt = mesure.meilleure.Count > 0 ? string.Join(" -> ", mesure.meilleure) : "aucune";
    Console.WriteLine($"Altruistes {groupe,-2} : cycles [{cyclesTxt}] + chaine {chaineTxt,-16} -> {mesure.servies.Count}/{paires.Length}");
}

var plan = Noyau03.CouvertureCyclesPuisChaine(paires, altruistes, 3, 4);
var serviesParCycles = new HashSet<string>();
foreach (var c in plan.cycles) foreach (var x in c) serviesParCycles.Add(x);
Console.WriteLine();
Console.WriteLine("Plan retenu (altruistes [" + string.Join(", ", altruistes.Select(a => "'" + a + "'")) + "]) :");
Console.WriteLine("  cycles : [" + string.Join(", ", plan.cycles.Select(c => "[" + string.Join(", ", c.Select(Noyau03.Tuple)) + "]")) + "] -> " + serviesParCycles.Count + " paires");
Console.WriteLine("  chaine : [" + string.Join(", ", plan.meilleure.Select(Noyau03.Tuple)) + "] -> orphelines servies : [" + string.Join(", ", plan.meilleure.Where(x => x.StartsWith("P") && !serviesParCycles.Contains(x)).OrderBy(x => x, StringComparer.Ordinal).Select(Noyau03.Tuple)) + "]");
Console.WriteLine("  couverture : " + plan.servies.Count + " / " + paires.Length + " | taux : " + (plan.servies.Count / (double)paires.Length).ToString("0.0###", CultureInfo.InvariantCulture));
Console.WriteLine("  paires non servies : [" + string.Join(", ", Enumerable.Range(0, paires.Length).Where(i => !plan.servies.Contains("P" + i)).Select(i => "('" + paires[i].donneur + "', '" + paires[i].receveur + "')")) + "]");

// Assertions : la couverture mesuree est celle de la source.
var couvertureAttendue = new Dictionary<string, int> { { "O", 4 }, { "A", 4 }, { "B", 6 }, { "AB", 5 } };
foreach (var groupe in groupes)
{
    var m = Noyau03.CouvertureCyclesPuisChaine(paires, new[] { groupe, groupe }, 3, 4);
    if (m.servies.Count != couvertureAttendue[groupe])
        throw new Exception("couverture inattendue pour " + groupe + " : " + m.servies.Count);
}
if (plan.servies.Count != 6) throw new Exception("couverture du plan retenu : " + plan.servies.Count);


Altruistes O  : cycles [P0,P2 | P1,P3] + chaine aucune           -> 4/6


Altruistes A  : cycles [P0,P2 | P1,P3] + chaine aucune           -> 4/6


Altruistes B  : cycles [P0,P2 | P1,P3] + chaine A0 -> P5 -> P4   -> 6/6


Altruistes AB : cycles [P0,P2 | P1,P3] + chaine A0 -> P4         -> 5/6


Plan retenu (altruistes ['B', 'B']) :


  cycles : [[('P', 0), ('P', 2)], [('P', 1), ('P', 3)]] -> 4 paires


  chaine : [('A', 0), ('P', 5), ('P', 4)] -> orphelines servies : [('P', 4), ('P', 5)]


  couverture : 6 / 6 | taux : 1.0


  paires non servies : []


### Lecture du résultat — ce que la chaîne déverrouille, et à quelle condition

La couverture par cycles seuls plafonne à **4/6** : deux boucles de deux paires (P0↔P2 et P1↔P3). Aucune boucle ne peut accueillir P4 — son receveur AB ne matche que les donneurs de P1 et P3, déjà engagés — ni P5, dont le receveur B n'est atteignable que par les donneurs de P0 et P2, eux aussi engagés.

La chaîne ne se mesure pas sur le graphe complet. Un donneur ne donne qu'une fois : une paire servie par un cycle est **hors jeu**, et l'énumération doit courir sur le graphe **résiduel**. C'est ce que fait la mesure ci-dessus, et le résultat dépend alors du seul paramètre qui varie, le **groupe du donneur altruiste** : `O` et `A` laissent la couverture à **4/6** — le seul chemin d'un altruiste `O` entre dans P1, déjà servie, et aucun receveur n'est de groupe `A` — tandis qu'`AB` la porte à **5/6** avec la chaîne `[A0 → P4]`, et `B` à **6/6** avec la chaîne `[A0 → P5 → P4]`, qui sert les deux orphelines d'un coup.

Ce que la chaîne déverrouille est donc une **liberté conditionnelle** : elle peut terminer sans refermer de boucle — le donneur de la dernière paire donne à la liste d'attente — mais cette liberté ne sert à rien si le groupe de l'altruiste n'atteint qu'une paire déjà engagée par les cycles. Sur cette instance, P5 n'est débloquée que par un altruiste `B`, et c'est alors P4 qui suit dans la même chaîne : un seul donneur altruiste, deux paires gagnées.

Le prix se lit dans cette dépendance : un donneur altruiste est rare, et son groupe décide de qui il débloque. La question d'arbitrage cardinalité/équité — quelles paires sacrifie-t-on quand la pool est trop petite — est celle de Coalitions-03, tranchée là-bas par le front de Pareto.

## 5. Expérience 3 — L'heuristique, l'optimum, le certificat

*Ancrée dans [Coalitions-04](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb) (dont l'algorithme est repris intégralement) et [Coalitions-05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) (dont le lake fournit le certificat).*

Trois réponses à « quelle est la meilleure affectation de n agents sur n tâches » : l'**heuristique** gloutonne (rapide, sans garantie), l'**optimum** de Kuhn-Munkres (garanti par le couple dual), et le **théorème** du lake (la garantie elle-même devient un objet machine-vérifié). Nous mettons les trois sur le **même objet** : une matrice 3×3, puis une série de matrices aléatoires seedées.

In [6]:
// Kuhn-Munkres repris INTEGRALEMENT de Coalitions-04 (methode hongroise pedagogique,
// arithmetique entiere exacte, chaque iteration construit une fois l'arbre hongrois).
var C3 = new long[3, 3] { { 4, 1, 3 }, { 2, 0, 5 }, { 3, 2, 2 } };
var resultat = Noyau04.KuhnMunkres(C3);
var glouton = Noyau04.Glouton(C3);

Console.WriteLine("Matrice C3 : [[4, 1, 3], [2, 0, 5], [3, 2, 2]]");
Console.WriteLine("Glouton  : affectation [" + string.Join(", ", glouton.affectation.Select(j => (j + 1).ToString(CultureInfo.InvariantCulture))) + "] -> valeur " + glouton.valeur);
Console.WriteLine("Munkres  : affectation [" + string.Join(", ", resultat.affectation.Select(j => (j + 1).ToString(CultureInfo.InvariantCulture))) + "] -> valeur " + resultat.valeur);
Console.WriteLine("Ecart glouton - optimum : " + (glouton.valeur - resultat.valeur));

// Assertions : l'affectation et la valeur mesurees sont celles de la source.
if (resultat.valeur != 5 || glouton.valeur != 5) throw new Exception("valeur d'affectation : " + resultat.valeur + " / " + glouton.valeur);
if (!resultat.affectation.SequenceEqual(new[] { 1, 0, 2 })) throw new Exception("affectation Munkres differente de la source");

public static class Noyau04
{
    /// Retourne (valeur, affectation, u, v) avec u_i + v_j <= c_ij pour tout (i, j).
    public static (int valeur, int[] affectation, long[] u, long[] v) KuhnMunkres(long[,] C)
    {
        int n = C.GetLength(0);
        var u = new long[n];
        var v = new long[n];
        for (int i = 0; i < n; i++)
        {
            long mini = C[i, 0];
            for (int j = 1; j < n; j++) if (C[i, j] < mini) mini = C[i, j];
            u[i] = mini;
        }

        var matchCol = new int[n];
        var matchRow = new int[n];
        for (int i = 0; i < n; i++) { matchCol[i] = -1; matchRow[i] = -1; }

        for (int i0 = 0; i0 < n; i0++)
        {
            while (matchRow[i0] == -1)
            {
                var decJ = new Dictionary<int, int>();
                var lignes = new HashSet<int> { i0 };
                var colonnes = new HashSet<int>();
                var file = new Queue<int>();
                file.Enqueue(i0);
                int jLibre = -1;

                while (file.Count > 0 && jLibre == -1)
                {
                    int i = file.Dequeue();
                    for (int j = 0; j < n; j++)
                    {
                        if (colonnes.Contains(j) || u[i] + v[j] != C[i, j]) continue;
                        decJ[j] = i;
                        colonnes.Add(j);
                        if (matchCol[j] == -1) { jLibre = j; break; }
                        lignes.Add(matchCol[j]);
                        file.Enqueue(matchCol[j]);
                    }
                }

                if (jLibre != -1)
                {
                    int j = jLibre;
                    int i = decJ[j];
                    while (true)
                    {
                        int jSuivant = matchRow[i];
                        matchCol[j] = i;
                        matchRow[i] = j;
                        if (i == i0) break;
                        j = jSuivant;
                        i = decJ[j];
                    }
                }
                else
                {
                    long delta = long.MaxValue;
                    foreach (int i in lignes)
                        for (int j = 0; j < n; j++)
                            if (!colonnes.Contains(j)) delta = Math.Min(delta, C[i, j] - u[i] - v[j]);
                    foreach (int i in lignes) u[i] += delta;
                    foreach (int j in colonnes) v[j] -= delta;
                }
            }
        }

        if (matchRow.Any(x => x < 0)) throw new Exception("pas un matching parfait");
        int valeur = 0;
        for (int i = 0; i < n; i++) valeur += (int)C[i, matchRow[i]];
        return (valeur, matchRow, u, v);
    }

    /// Glouton du CAPSTONE (a comparer a l'exercice 2 de 04) : chaque ligne prend
    /// sa colonne libre la moins chere, dans l'ordre.
    public static (int[] affectation, int valeur) Glouton(long[,] C)
    {
        int n = C.GetLength(0);
        var libres = Enumerable.Range(0, n).ToList();
        var affectation = new int[n];
        for (int i = 0; i < n; i++)
        {
            int j = libres.MinBy(k => C[i, k]);
            libres.Remove(j);
            affectation[i] = j;
        }
        int valeur = 0;
        for (int i = 0; i < n; i++) valeur += (int)C[i, affectation[i]];
        return (affectation, valeur);
    }
}


Matrice C3 : [[4, 1, 3], [2, 0, 5], [3, 2, 2]]


Glouton  : affectation [2, 1, 3] -> valeur 5


Munkres  : affectation [2, 1, 3] -> valeur 5


Ecart glouton - optimum : 0


In [7]:
// L'ecart systematique : 500 matrices aleatoires seedees, 4x4.
// Attention : le flux de NumPy (PCG64) n'est pas reproductible en C#, on utilise
// donc System.Random(2026) -- l'ecart mesure est sur la meme experience (500
// matrices 4x4, memes bornes), pas sur le meme tir. La divergence de tirage est
// documentee dans le registre des jumeaux (known_differences).
var rng = new Random(2026);
var ecarts = new List<int>();
for (int k = 0; k < 500; k++)
{
    var M = new long[4, 4];
    for (int i = 0; i < 4; i++)
        for (int j = 0; j < 4; j++) M[i, j] = rng.Next(0, 20);
    var g = Noyau04.Glouton(M);
    var m = Noyau04.KuhnMunkres(M);
    ecarts.Add(g.valeur - m.valeur);
}

Console.WriteLine("500 matrices 4x4 (tirage System.Random(2026), propre au portage) :");
Console.WriteLine("  ecart moyen glouton - optimum : " + ecarts.Average().ToString("0.0###", CultureInfo.InvariantCulture));
Console.WriteLine("  ecart max : " + ecarts.Max() + " | matrices ou le glouton est optimal : " + ecarts.Count(e => e == 0) + " / 500");

// Assertions : le glouton n'est jamais meilleur que l'optimum, et l'ecart est borne.
if (ecarts.Any(e => e < 0)) throw new Exception("ecart negatif : le glouton aurait batu l'optimum");
if (ecarts.Max() > 60) throw new Exception("ecart max hors bornes attendues : " + ecarts.Max());


500 matrices 4x4 (tirage System.Random(2026), propre au portage) :


  ecart moyen glouton - optimum : 4.604


  ecart max : 26 | matrices ou le glouton est optimal : 208 / 500


In [8]:
// Le CERTIFICAT : pourquoi croire Munkres ? Le couple dual rendu par l'algorithme.
// Realisabilite duale : u_i + v_j <= c_ij partout ; valeur duale = somme(u) + somme(v).
// Si valeur duale = valeur primaire, l'ecart de dualite est nul : optimalite, le
// schema EXACT du theoreme Assignment.optimality_of_zero_gap du lake (Coalitions-05).
bool dualOk = true;
for (int i = 0; i < 3; i++)
    for (int j = 0; j < 3; j++)
        if (resultat.u[i] + resultat.v[j] > C3[i, j]) dualOk = false;
long dualVal = resultat.u.Sum() + resultat.v.Sum();

Console.WriteLine("Potentiels duaux u : [" + string.Join(", ", resultat.u) + "] | v : [" + string.Join(", ", resultat.v) + "]");
Console.WriteLine("Realisabilite duale (u_i + v_j <= c_ij partout) : " + (dualOk ? "True" : "False"));
Console.WriteLine("Valeur duale somme(u)+somme(v) : " + dualVal + " | valeur primaire Munkres : " + resultat.valeur);
Console.WriteLine("Ecart de dualite : " + (resultat.valeur - dualVal) + " -> " + (resultat.valeur == dualVal ? "optimalite certifiee" : "ANOMALIE"));

// Ce que le lake prouve, lu depuis ses sources (jamais execute ici — cf. 05) :
var racineLake = Path.Combine(racine.FullName, "MyIA.AI.Notebooks", "GameTheory", "assignment_lean", "Assignment", "KuhnMunkres.lean");
if (File.Exists(racineLake))
{
    var source = File.ReadAllText(racineLake);
    var exposes = source.Split('\n')
        .Where(l => l.StartsWith("theorem ") || l.StartsWith("lemma ") || l.StartsWith("def "))
        .Select(l => l.Split(new[] { ' ', '\t' }, StringSplitOptions.RemoveEmptyEntries)[1])
        .ToList();
    Console.WriteLine("API exposee par KuhnMunkres.lean : [" + string.Join(", ", exposes.Select(e => "'" + e + "'")) + "]");
}

// Assertions : le certificat affiche est celui de la source.
if (!dualOk) throw new Exception("realisabilite duale rompue");
if (resultat.valeur != dualVal) throw new Exception("ecart de dualite non nul : " + (resultat.valeur - dualVal));


Potentiels duaux u : [3, 2, 2] | v : [0, -2, 0]


Realisabilite duale (u_i + v_j <= c_ij partout) : True


Valeur duale somme(u)+somme(v) : 5 | valeur primaire Munkres : 5


Ecart de dualite : 0 -> optimalite certifiee


API exposee par KuhnMunkres.lean : ['EqEdge', 'kuhn_munkres_correct', 'dualFeasible_tighten']


### Lecture du résultat — trois garanties sur un seul objet

Sur C3, le glouton tombe sur l'optimum par chance (écart 0) — c'est possible, jamais garanti : la loterie se mesure sur les 500 matrices seedées, où le glouton n'est optimal que **208 fois sur 500**, avec un écart moyen de **4,604** et un pire cas à **26**. (Le tirage C# passe par `System.Random(2026)` : ce n'est pas le flux de NumPy, donc ces nombres diffèrent de ceux du jumeau Python — même expérience, valeurs distinctes.) Munkres, lui, **rend la preuve avec le résultat** : le couple `(u, v)` est réalisable (`u_i + v_j <= c_ij`, vérifié ci-dessus en une ligne), et sa somme égale la valeur de l'affectation — écart de dualité nul, donc optimalité, par le théorème de dualité faible que le lake `assignment_lean` énonce (`Assignment.weak_duality`) et dont `Assignment.optimality_of_zero_gap` fait le critère. C'est le saut de garantie : le glouton *affirme*, Munkres *certifie*, le lake *démontre le certificat*. L'inventaire complet du lake fait l'objet de la section suivante.


## 6. Ce que le lake Lean prouve — inventaire mesuré

Le lake `assignment_lean` vit sous `MyIA.AI.Notebooks/GameTheory/assignment_lean/`. Nous l'inventorions **par lecture directe de ses sources** (le pattern de Coalitions-03 et 05) : modules, théorèmes, définitions, `sorry`. Le lake n'est jamais invoqué ici — sa preuve de build vit dans la CI de Coalitions-05.

In [9]:
// Inventaire par lecture directe des sources (pattern Coalitions-03/05, sans lake).
var lake = Path.Combine(racine.FullName, "MyIA.AI.Notebooks", "GameTheory", "assignment_lean", "Assignment");
var lignesInventaire = new List<string[]>();
int totalThm = 0, totalDef = 0, totalSorry = 0;
foreach (var fichier in Directory.GetFiles(lake, "*.lean").OrderBy(p => p, StringComparer.Ordinal))
{
    string nom = Path.GetFileNameWithoutExtension(fichier);
    if (nom.EndsWith("_en")) continue;  // jumeaux de traduction — comptes a part
    string src = File.ReadAllText(fichier);
    // commentaire-strip AVANT de compter sorry (cf. regle anti-regression : jamais grep -c sorry)
    string sansCommentaires = Regex.Replace(src, @"/-.*?-/", "", RegexOptions.Singleline);
    sansCommentaires = Regex.Replace(sansCommentaires, @"--.*$", "", RegexOptions.Multiline);
    int nThm = Regex.Matches(src, @"^(?:theorem|lemma)\s+\w+", RegexOptions.Multiline).Count;
    int nDef = Regex.Matches(src, @"^def\s+\w+", RegexOptions.Multiline).Count;
    int nSorry = Regex.Matches(sansCommentaires, @"^\s*sorry\b", RegexOptions.Multiline).Count;
    totalThm += nThm; totalDef += nDef; totalSorry += nSorry;
    lignesInventaire.Add(new[] { nom, nThm.ToString(CultureInfo.InvariantCulture), nDef.ToString(CultureInfo.InvariantCulture), nSorry.ToString(CultureInfo.InvariantCulture) });
}

foreach (var l in Table.RendreTable(new[] { "module", "theoremes", "defs", "sorry" }, lignesInventaire.ToArray(), 2, null, null))
    Console.WriteLine(l);
int jumeauxEn = Directory.GetFiles(lake, "*_en.lean").Length;
Console.WriteLine($"Totaux FR : {totalThm} theoremes, {totalDef} definitions, {totalSorry} sorry | jumeaux _en : {jumeauxEn}");

// Assertion : l'inventaire mesure est celui de la source.
if (totalThm != 9 || totalDef != 7 || totalSorry != 0 || jumeauxEn != 5)
    throw new Exception($"inventaire inattendu : {totalThm}/{totalDef}/{totalSorry}, jumeaux {jumeauxEn}");


     module  theoremes  defs  sorry


   Counting          4     2      0


Definitions          0     2      0


    Duality          1     2      0


KuhnMunkres          2     1      0


 Optimality          2     0      0


Totaux FR : 9 theoremes, 7 definitions, 0 sorry | jumeaux _en : 5


### Lecture de l'inventaire — la colonne vertébrale de la garantie

L'inventaire mesuré ci-dessus donne la structure de la preuve : les **définitions** posent les objets (`Assignment.value`, la valeur d'une affectation ; `DualFeasible`, la réalisabilité duale), les **théorèmes** enchaînent les garanties — `weak_duality` (toute affectation vaut au plus tout dual réalisable), `optimality_of_zero_gap` (écart nul = optimalité : le critère invoqué à l'expérience 3), `kuhn_munkres_correct` (l'algorithme lui-même, prouvé) — et la colonne `sorry` à **zéro** dit qu'aucune maille n'est laissée en trémie : ce qui est énoncé est démontré. Chaque module FR a son jumeau `_en` byte-identique hors commentaires (la convention i18n du lac). C'est le niveau de garantie terminal : pas « l'algorithme est correct » affirmé, mais **démontré par un noyau qui ne croit que ce qu'il vérifie** — et ce noyau, c'est la CI de [Coalitions-05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) qui le relit à chaque changement.

## 7. Par où y entrer

Trois parcours dans la sous-série, selon le temps et la question :

- **Une heure, le pouvoir** : [Coalitions-01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) seul — les trois indices, la seconde voie SMT, la cross-validation sur neuf couples. Il referme la question « poids contre pouvoir » ouverte par l'expérience 1.
- **Deux heures, l'institution** : [01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) puis [03](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb) — du pouvoir mesuré à l'arbitrage cardinalité/équité d'un programme d'échanges réel, dont l'expérience 2 a montré le levier altruiste.
- **La boucle entière** : [01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) → [04](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb) → [05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) — l'affectation depuis l'heuristique jusqu'au théorème machine-vérifié, bouclée par [02](Coalitions/Coalitions-02-Assistance-Games-Python.ipynb) qui replace la coopération dans le temps (bornes no-regret).

## 8. Exercices

Trois exercices non résolus, un par palier de garantie. Les stubs s'exécutent sans erreur ; à vous de les remplir.

In [10]:
// Exercice 1 (suite directe de 01, exercice 2) : le jeu de rupture.
// Un jeu ou un joueur de poids MINIMAL detient un pouvoir de veto mesurable.
// TODO etudiant :
// Etape 1 : construire un VotingGame de la forme [q; 1, 1, 1, 1, w] ou le petit
//           poids w fait de son porteur un joueur veto (indice : les quatre autres
//           ensemble ne font quota que si q > 4).
// Etape 2 : mesurer Noyau01.Banzhaf(jeu) et verifier que is_veto... — le capstone
//           n'a garde que IsDummy ; ajoutez le predicat IsVeto du carnet 01.
// Etape 3 : interpreter : quel est le Banzhaf d'un veto, et pourquoi ?
// Le message final est identique a celui du carnet Python (parite des sorties).
void JeuDeRupture()
{
    // TODO etudiant : retourner le VotingGame et le dictionnaire banzhaf mesure.
    Console.WriteLine("Exercice 1 a completer : jeu_de_rupture()");
}

JeuDeRupture();


Exercice 1 a completer : jeu_de_rupture()


In [11]:
// Exercice 2 (a partir de 02) : le budget de l'apprenant.
// EXP3 garantit un regret en racine(T) ; que reste-t-il avec un budget d'essais borne ?
// TODO etudiant :
// Etape 1 : relire la boucle EXP3 de Coalitions-02 (section bench).
// Etape 2 : la borne a T/2 essais effectifs et mesurer le regret empirique moyen
//           sur 200 repetitions (meme oracle hindsight que 02).
// Etape 3 : comparer a la borne theorique en racine(T) : la perte est-elle en
//           constante multiplicative ou en ordre de grandeur ?
// Le message final est identique a celui du carnet Python (parite des sorties).
void RegretBudget()
{
    // TODO etudiant : retourner (regret_complet, regret_budget).
    Console.WriteLine("Exercice 2 a completer : regret_budget()");
}

RegretBudget();


Exercice 2 a completer : regret_budget()


In [12]:
// Exercice 3 (04/05) : l'instance ou le glouton egale Munkres — et pourquoi le certificat reste du.
// TODO etudiant :
// Etape 1 : reprendre la boucle de l'experience 3 et isoler les matrices 3x3 OU le
//           glouton est optimal (ecart nul) — il en existe (mesure ci-dessus).
// Etape 2 : pour l'une d'elles, executer Noyau04.KuhnMunkres et verifier le certificat
//           dual (realisabilite + ecart nul), comme dans la cellule du certificat.
// Etape 3 : expliquer en une phrase : si le glouton est optimal sur CETTE matrice,
//           qu'est-ce que le certificat ajoute quand meme que la valeur egaleie ne prouve pas ?
//           (indice : une egalite mesuree sur une instance n'est pas une garantie sur toutes.)
// Le message final est identique a celui du carnet Python (parite des sorties).
void GloutonOptimalInstance()
{
    // TODO etudiant : retourner (M, valeur_glouton, valeur_munkres, certificat_ok).
    Console.WriteLine("Exercice 3 a completer : glouton_optimal_instance()");
}

GloutonOptimalInstance();


Exercice 3 a completer : glouton_optimal_instance()


## À retenir

La sous-série Coalitions répond à « combien vaut une coalition » par une **échelle de garanties**, et le capstone a fait monter trois de ses barreaux sur les mêmes objets :

1. **Mesuré** — le pouvoir contrefactuel (Banzhaf) : un vote de quota de plus et le président à 60 % du poids tombe à égalité de pouvoir, le troisième devient nul. La valeur d'un membre est dans ses swings, pas dans son poids ([01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb)).
2. **Institutionnel** — la chaîne altruiste déverrouille ce que le cycle ne peut pas refermer, à condition que son groupe atteigne une paire restée libre : 4/6 sans chaîne, 6/6 avec un altruiste `B` sur l'instance ; l'arbitrage cardinalité/équité reste ([03](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb)).
3. **Certifié, démontré** — le glouton mesure, Munkres certifie (`u_i + v_j <= c_ij` + écart nul), le lake démontre le critère du certificat, sans un `sorry` ([04](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb), [05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb)).

Et entre les deux, [02](Coalitions/Coalitions-02-Assistance-Games-Python.ipynb) rappelle que la coopération a aussi un prix temporel : la borne no-regret est le niveau de garantie de l'apprenant qui n'a pas d'oracle. Cinq carnets, une échelle : chaque fois que la question « est-ce prouvé ? » se pose, la sous-série a un barreau à offrir.